# Video Generation

## Import library

In [41]:
import cv2
import numpy as np
import os
import random

In [42]:
input_folder = "../Preprocessed_Dataset"

classes = [
    folder for folder in os.listdir(input_folder)
    if os.path.isdir(os.path.join(input_folder, folder))
]

print(classes)

['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']


In [43]:
images = []

for cls in classes:
    folder = os.path.join(input_folder, cls)

    for file in os.listdir(folder):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            images.append(os.path.join(folder, file))

print("Total Images:", len(images))

selected = random.sample(images, 50)

Total Images: 693


In [44]:
video_name = "PCB_Conveyor.mp4"

width = 1280
height = 720

fps = 60

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

video = cv2.VideoWriter(
    video_name,
    fourcc,
    fps,
    (width, height)
)

In [45]:
pcb_images = []

for img_path in selected:

    img = cv2.imread(img_path)

    if img is None:
        continue

    img = cv2.resize(img, (180,180))

    pcb_images.append(img)

In [46]:
positions = []

spacing = 250

for i in range(len(pcb_images)):

    x = -i * spacing

    belt_y = 270

    positions.append([x, belt_y])

In [47]:
speed = 5

total_frames = 1200

for frame_id in range(total_frames):

    frame = np.full((height, width, 3), 235, dtype=np.uint8)

    cv2.rectangle(  
        frame,
        (0,220),
        (width,500),
        (100,100,100),
        -1
    )

    for i, pcb in enumerate(pcb_images):

        x, y = positions[i]

        x += speed

        positions[i][0] = x

        if x > width:
            positions[i][0] = -200
            continue

        if x + 180 < 0:
            continue

        start_x = max(0, x)
        end_x = min(width, x + 180)

        pcb_start = max(0, -x)
        pcb_end = pcb_start + (end_x - start_x)

        frame[
            y:y+180,
            start_x:end_x
        ] = pcb[
            :,
            pcb_start:pcb_end
        ]

    video.write(frame)

In [48]:
video.release()

print("Video Saved!")

Video Saved!
